# Reto Semana 4: Sistema de Inventario Modular

## Programacion para Ciencia de Datos
**Instituto Politecnico Nacional** | Semestre Febrero-Julio 2026

---

## El Escenario del Mundo Real

Eres desarrollador en una cadena de tiendas de tecnologia. El gerente necesita un sistema que:

1. **Lea** el inventario de un archivo CSV
2. **Identifique** productos que necesitan reorden (stock bajo)
3. **Genere** un reporte para el departamento de compras

El codigo debe estar **organizado profesionalmente** para que otros desarrolladores puedan mantenerlo y extenderlo.

```
╔════════════════════════════════════════════════════════════════════════════╗
║                  SISTEMA DE INVENTARIO MODULAR                             ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║   inventario.csv                                                           ║
║   ┌────────────────────────────────────────┐                               ║
║   │ SKU001,Laptop,Electronica,15000,5,10   │                               ║
║   │ SKU002,Mouse,Accesorios,350,3,15       │  <- Stock < Minimo!           ║
║   │ SKU003,Teclado,Accesorios,800,20,10    │                               ║
║   └────────────────────────────────────────┘                               ║
║                      │                                                     ║
║                      ▼                                                     ║
║   ┌─────────────────────────────────────────────────────────────────┐      ║
║   │                    TU SISTEMA MODULAR                           │      ║
║   │  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐           │      ║
║   │  │   models/    │  │    utils/    │  │    main.py   │           │      ║
║   │  │ producto.py  │  │ validators   │  │  Orquesta    │           │      ║
║   │  │ Clase        │  │ io.py        │  │  todo        │           │      ║
║   │  └──────────────┘  └──────────────┘  └──────────────┘           │      ║
║   └─────────────────────────────────────────────────────────────────┘      ║
║                      │                                                     ║
║                      ▼                                                     ║
║   reporte_inventario.csv                                                   ║
║   ┌────────────────────────────────────────────────────────────────┐       ║
║   │ SKU002,Mouse,Accesorios,3,15,12,1050.00  <- Necesita reorden   │       ║
║   └────────────────────────────────────────────────────────────────┘       ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
```

---

## Estructura de Proyecto Requerida

**IMPORTANTE**: Este reto requiere una estructura de carpetas especifica.

```
reto_semana_04/
│
├── main.py                    # Punto de entrada
├── README.md                  # Documentacion
├── .gitignore                 # Archivos a ignorar
│
├── models/                    # Clases de dominio
│   ├── __init__.py
│   └── producto.py            # Clase Producto
│
├── utils/                     # Utilidades
│   ├── __init__.py
│   ├── io.py                  # Funciones de lectura/escritura
│   └── validators.py          # Funciones de validacion
│
├── data/                      # Datos de entrada
│   └── inventario.csv
│
└── outputs/                   # Resultados
    └── reporte_inventario.csv
```

### Contenido de los archivos `__init__.py`

In [ ]:
# models/__init__.py
models_init = '''
"""Modelos de dominio del sistema de inventario."""
from .producto import Producto
'''

# utils/__init__.py
utils_init = '''
"""Utilidades del sistema de inventario."""
from .validators import validar_sku, validar_precio, validar_stock
from .io import leer_inventario, escribir_reporte
'''

print("=== models/__init__.py ===")
print(models_init)
print("\n=== utils/__init__.py ===")
print(utils_init)

---

## Especificacion de Entrada

### Archivo: `data/inventario.csv`

**IMPORTANTE**: Cada producto aparece **una sola vez** en el archivo de entrada, identificado por su SKU unico. **No hay registros duplicados** para el mismo producto, por lo tanto **no se requiere consolidacion ni agrupacion**. Cada linea representa el estado actual de un producto en el inventario.

### Columnas

| Columna | Tipo | Descripcion | Ejemplo |
|---------|------|-------------|----------|
| `sku` | texto | Identificador unico del producto | `SKU001` |
| `nombre` | texto | Nombre del producto | `Laptop HP` |
| `categoria` | texto | Categoria del producto | `Electronica` |
| `precio` | decimal | Precio unitario | `15000.00` |
| `stock` | entero | Cantidad actual en inventario | `5` |
| `stock_minimo` | entero | Nivel minimo antes de reordenar | `10` |

### Ejemplo de Entrada Valida

```csv
sku,nombre,categoria,precio,stock,stock_minimo
SKU001,Laptop HP,Electronica,15000.00,5,10
SKU002,Mouse Logitech,Accesorios,350.00,3,15
SKU003,Teclado Mecanico,Accesorios,800.00,20,10
SKU004,Monitor LG,Electronica,6000.00,8,5
SKU005,Audifonos Sony,Accesorios,1200.00,2,10
SKU006,Webcam HD,Accesorios,450.00,25,20
SKU007,SSD 1TB,Almacenamiento,1800.00,0,5
```

### Posibles Errores en los Datos

El archivo de entrada **puede contener lineas con errores**. Tu programa debe detectarlos e **ignorar las lineas invalidas** sin detener la ejecucion. Los tipos de errores posibles son:

#### Error Tipo 1: Precio no numerico
El campo `precio` contiene texto en lugar de un numero valido.

```csv
SKU010,Cable HDMI,Cables,N/A,12,5
SKU011,Router WiFi,Redes,pendiente,8,10
SKU012,Mousepad,Accesorios,00,30,20
```

#### Error Tipo 2: Stock no numerico
El campo `stock` contiene texto en lugar de un entero valido.

```csv
SKU013,Webcam 4K,Video,2500.00,abc,10
SKU014,SSD Externo,Almacenamiento,1800.00,null,5
SKU015,Teclado Gaming,Accesorios,1200.00,muchos,15
```

#### Error Tipo 3: Stock minimo no numerico
El campo `stock_minimo` contiene texto en lugar de un entero valido.

```csv
SKU016,Monitor Curvo,Electronica,8000.00,12,???
SKU017,Audifonos BT,Audio,600.00,5,sin_dato
```

#### Error Tipo 4: Columnas faltantes
La linea tiene **menos de 6 columnas** (faltan uno o mas campos).

```csv
SKU018,Impresora,Impresion
SKU019,UPS
SKU020,Bocina,Audio,500.00,8
```

#### Error Tipo 5: Columnas extra
La linea tiene **mas de 6 columnas** (campos adicionales no esperados).

```csv
SKU021,Mouse Gaming,Accesorios,800.00,15,10,extra1,extra2
SKU022,Tablet,Electronica,7000.00,3,5,sobrante
```

### Ejemplo de Entrada con Errores

```csv
sku,nombre,categoria,precio,stock,stock_minimo
SKU001,Laptop HP,Electronica,15000.00,5,10
SKU002,Mouse Logitech,Accesorios,350.00,3,15
SKU003,Teclado Mecanico,Accesorios,N/A,20,10
SKU004,Monitor LG,Electronica,6000.00,abc,5
SKU005,Audifonos Sony,Accesorios,1200.00,2,???
SKU006,Webcam HD,Accesorios
SKU007,SSD 1TB,Almacenamiento,1800.00,0,5,dato_extra,otro
```

En este ejemplo, solo SKU001, SKU002 y posiblemente otros registros validos deben procesarse. Las lineas de SKU003-SKU007 deben **ignorarse silenciosamente** o reportar una advertencia.

---

## Componente 1: Clase Producto

### Archivo: `models/producto.py`

Debes crear una clase `Producto` con los siguientes requisitos:

In [ ]:
class Producto:
    """
    Representa un producto en el inventario.
    
    Attributes:
        sku (str): Identificador unico
        nombre (str): Nombre del producto
        categoria (str): Categoria del producto
        precio (float): Precio unitario
        stock (int): Cantidad actual
        stock_minimo (int): Nivel minimo de stock
    """
    
    def __init__(self, sku, nombre, categoria, precio, stock, stock_minimo):
        """
        Inicializa un nuevo producto.
        
        Args:
            sku: Identificador unico del producto
            nombre: Nombre del producto
            categoria: Categoria del producto
            precio: Precio unitario (debe ser >= 0)
            stock: Cantidad actual (debe ser >= 0)
            stock_minimo: Nivel minimo de stock (debe ser >= 0)
        """
        self.sku = sku
        self.nombre = nombre
        self.categoria = categoria
        self.precio = precio
        self.stock = stock
        self.stock_minimo = stock_minimo
    
    def necesita_reorden(self):
        """
        Determina si el producto necesita ser reordenado.
        
        Returns:
            bool: True si stock < stock_minimo, False en caso contrario
        """
        return self.stock < self.stock_minimo
    
    def unidades_faltantes(self):
        """
        Calcula cuantas unidades faltan para alcanzar el stock minimo.
        
        Returns:
            int: Unidades faltantes (0 si no necesita reorden)
        """
        if self.necesita_reorden():
            return self.stock_minimo - self.stock
        return 0
    
    def valor_inventario(self):
        """
        Calcula el valor monetario del inventario actual.
        
        Returns:
            float: precio * stock
        """
        return self.precio * self.stock
    
    def __str__(self):
        """Representacion legible para usuarios."""
        estado = "[REORDEN]" if self.necesita_reorden() else "[OK]"
        return f"{estado} {self.sku}: {self.nombre} - Stock: {self.stock}/{self.stock_minimo}"
    
    def __repr__(self):
        """Representacion tecnica para desarrolladores."""
        return (f"Producto('{self.sku}', '{self.nombre}', '{self.categoria}', "
                f"{self.precio}, {self.stock}, {self.stock_minimo})")

In [ ]:
# Pruebas de la clase Producto

# Crear productos de prueba
laptop = Producto("SKU001", "Laptop HP", "Electronica", 15000.00, 5, 10)
mouse = Producto("SKU002", "Mouse Logitech", "Accesorios", 350.00, 20, 15)

print("=== Laptop (necesita reorden) ===")
print(f"str: {laptop}")
print(f"repr: {repr(laptop)}")
print(f"necesita_reorden(): {laptop.necesita_reorden()}")
print(f"unidades_faltantes(): {laptop.unidades_faltantes()}")
print(f"valor_inventario(): ${laptop.valor_inventario():,.2f}")

print("\n=== Mouse (OK) ===")
print(f"str: {mouse}")
print(f"necesita_reorden(): {mouse.necesita_reorden()}")
print(f"unidades_faltantes(): {mouse.unidades_faltantes()}")

---

## Componente 2: Validadores

### Archivo: `utils/validators.py`

Funciones para validar datos antes de crear objetos Producto.

In [ ]:
# utils/validators.py

def validar_sku(sku):
    """
    Valida que el SKU no este vacio.
    
    Args:
        sku: El SKU a validar
        
    Returns:
        bool: True si es valido, False si no
    """
    if not sku or not str(sku).strip():
        return False
    return True


def validar_precio(precio):
    """
    Valida que el precio sea un numero >= 0.
    
    Args:
        precio: El precio a validar
        
    Returns:
        bool: True si es valido, False si no
    """
    try:
        precio_num = float(precio)
        return precio_num >= 0
    except (ValueError, TypeError):
        return False


def validar_stock(stock):
    """
    Valida que el stock sea un entero >= 0.
    
    Args:
        stock: El stock a validar
        
    Returns:
        bool: True si es valido, False si no
    """
    try:
        stock_num = int(stock)
        return stock_num >= 0
    except (ValueError, TypeError):
        return False


def validar_producto(sku, nombre, categoria, precio, stock, stock_minimo):
    """
    Valida todos los campos de un producto.
    
    Returns:
        tuple: (es_valido: bool, mensaje_error: str o None)
    """
    if not validar_sku(sku):
        return False, "SKU vacio o invalido"
    
    if not nombre or not str(nombre).strip():
        return False, "Nombre vacio"
    
    if not validar_precio(precio):
        return False, f"Precio invalido: {precio}"
    
    if not validar_stock(stock):
        return False, f"Stock invalido: {stock}"
    
    if not validar_stock(stock_minimo):
        return False, f"Stock minimo invalido: {stock_minimo}"
    
    return True, None


# Pruebas
print("=== Pruebas de Validadores ===")
print(f"validar_sku('SKU001'): {validar_sku('SKU001')}")
print(f"validar_sku(''): {validar_sku('')}")
print(f"validar_sku(None): {validar_sku(None)}")

print(f"\nvalidar_precio(100.50): {validar_precio(100.50)}")
print(f"validar_precio(-10): {validar_precio(-10)}")
print(f"validar_precio('abc'): {validar_precio('abc')}")

print(f"\nvalidar_stock(10): {validar_stock(10)}")
print(f"validar_stock(-5): {validar_stock(-5)}")
print(f"validar_stock('invalid'): {validar_stock('invalid')}")

---

## Componente 3: Funciones de I/O

### Archivo: `utils/io.py`

Funciones para leer el CSV de entrada y escribir el reporte.

In [ ]:
# utils/io.py

def leer_inventario(ruta_archivo):
    """
    Lee el archivo de inventario y retorna una lista de diccionarios.
    
    Args:
        ruta_archivo: Ruta al archivo CSV
        
    Returns:
        list: Lista de diccionarios con los datos de cada producto
        
    Raises:
        FileNotFoundError: Si el archivo no existe
    """
    productos_raw = []
    
    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        lineas = archivo.readlines()
        
        # Primera linea son los encabezados
        if not lineas:
            return productos_raw
        
        encabezados = lineas[0].strip().split(',')
        
        # Procesar cada linea de datos
        for linea in lineas[1:]:
            linea = linea.strip()
            if not linea:
                continue
            
            valores = linea.split(',')
            if len(valores) == len(encabezados):
                producto_dict = dict(zip(encabezados, valores))
                productos_raw.append(producto_dict)
    
    return productos_raw


def escribir_reporte(productos, ruta_archivo):
    """
    Escribe el reporte de productos que necesitan reorden.
    
    Args:
        productos: Lista de objetos Producto
        ruta_archivo: Ruta donde guardar el CSV
    """
    encabezados = [
        "sku", "nombre", "categoria", "stock_actual", 
        "stock_minimo", "unidades_faltantes", "valor_inventario"
    ]
    
    with open(ruta_archivo, 'w', encoding='utf-8') as archivo:
        # Escribir encabezados
        archivo.write(','.join(encabezados) + '\n')
        
        # Escribir datos
        for p in productos:
            linea = f"{p.sku},{p.nombre},{p.categoria},{p.stock},"
            linea += f"{p.stock_minimo},{p.unidades_faltantes()},{p.valor_inventario():.2f}"
            archivo.write(linea + '\n')


# Simulacion para el notebook
print("Funciones de I/O definidas:")
print("- leer_inventario(ruta): Lee CSV y retorna lista de dicts")
print("- escribir_reporte(productos, ruta): Escribe CSV de reporte")

---

## Componente 4: Programa Principal

### Archivo: `main.py`

In [ ]:
main_py = '''
#!/usr/bin/env python3
"""
Sistema de Inventario Modular
Genera reporte de productos que necesitan reorden.
"""

from models.producto import Producto
from utils.validators import validar_producto
from utils.io import leer_inventario, escribir_reporte

# Configuracion
ARCHIVO_INVENTARIO = "data/inventario.csv"
ARCHIVO_REPORTE = "outputs/reporte_inventario.csv"


def crear_productos(datos_raw):
    """
    Convierte lista de diccionarios en objetos Producto.
    Ignora registros invalidos.
    """
    productos = []
    
    for datos in datos_raw:
        # Validar
        es_valido, error = validar_producto(
            datos.get('sku'),
            datos.get('nombre'),
            datos.get('categoria'),
            datos.get('precio'),
            datos.get('stock'),
            datos.get('stock_minimo')
        )
        
        if not es_valido:
            print(f"Advertencia: Ignorando registro invalido - {error}")
            continue
        
        # Crear objeto Producto
        producto = Producto(
            sku=datos['sku'],
            nombre=datos['nombre'],
            categoria=datos['categoria'],
            precio=float(datos['precio']),
            stock=int(datos['stock']),
            stock_minimo=int(datos['stock_minimo'])
        )
        productos.append(producto)
    
    return productos


def filtrar_necesitan_reorden(productos):
    """Filtra productos que necesitan reorden."""
    return [p for p in productos if p.necesita_reorden()]


def ordenar_por_faltantes(productos):
    """Ordena por unidades faltantes (descendente)."""
    return sorted(productos, key=lambda p: p.unidades_faltantes(), reverse=True)


def main():
    print("=" * 50)
    print("SISTEMA DE INVENTARIO - Reporte de Reorden")
    print("=" * 50)
    
    # 1. Leer datos
    print(f"\nLeyendo inventario de: {ARCHIVO_INVENTARIO}")
    datos_raw = leer_inventario(ARCHIVO_INVENTARIO)
    print(f"Registros leidos: {len(datos_raw)}")
    
    # 2. Crear objetos Producto
    productos = crear_productos(datos_raw)
    print(f"Productos validos: {len(productos)}")
    
    # 3. Filtrar los que necesitan reorden
    necesitan_reorden = filtrar_necesitan_reorden(productos)
    print(f"Productos que necesitan reorden: {len(necesitan_reorden)}")
    
    # 4. Ordenar por unidades faltantes
    necesitan_reorden = ordenar_por_faltantes(necesitan_reorden)
    
    # 5. Mostrar resumen
    print("\n" + "-" * 50)
    print("PRODUCTOS QUE NECESITAN REORDEN:")
    print("-" * 50)
    for p in necesitan_reorden:
        print(p)
    
    # 6. Escribir reporte
    escribir_reporte(necesitan_reorden, ARCHIVO_REPORTE)
    print(f"\nReporte guardado en: {ARCHIVO_REPORTE}")
    
    print("\n" + "=" * 50)
    print("Proceso completado exitosamente")
    print("=" * 50)


if __name__ == "__main__":
    main()
'''

print(main_py)

---

## Especificacion de Salida

### Archivo: `outputs/reporte_inventario.csv`

El reporte debe contener **solo productos que necesitan reorden** (`stock < stock_minimo`).

**IMPORTANTE**: Como cada producto aparece una sola vez en la entrada (identificado por SKU unico), **no se requiere hacer agregaciones ni consolidacion**. Cada producto valido de la entrada que cumpla la condicion de reorden aparecera como una sola linea en la salida.

### Columnas de Salida

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| `sku` | texto | SKU del producto |
| `nombre` | texto | Nombre del producto |
| `categoria` | texto | Categoria |
| `stock_actual` | entero | Stock actual |
| `stock_minimo` | entero | Stock minimo requerido |
| `unidades_faltantes` | entero | `stock_minimo - stock_actual` |
| `valor_inventario` | decimal (2 dec) | `precio * stock_actual` |

### Ordenamiento

Ordenar por `unidades_faltantes` **descendente** (los que necesitan mas unidades primero).

### Ejemplo de Salida

Dada la entrada valida del ejemplo anterior (SKU001-SKU007 sin errores):

```csv
sku,nombre,categoria,stock_actual,stock_minimo,unidades_faltantes,valor_inventario
SKU002,Mouse Logitech,Accesorios,3,15,12,1050.00
SKU005,Audifonos Sony,Accesorios,2,10,8,2400.00
SKU001,Laptop HP,Electronica,5,10,5,75000.00
SKU007,SSD 1TB,Almacenamiento,0,5,5,0.00
```

Nota: SKU003 (stock 20 >= minimo 10), SKU004 (stock 8 >= minimo 5) y SKU006 (stock 25 >= minimo 20) **no aparecen** en la salida porque no necesitan reorden.

---

## Ejemplo Completo de Ejecucion

In [ ]:
# Simulacion completa del sistema

# Datos de entrada simulados
datos_inventario = [
    {"sku": "SKU001", "nombre": "Laptop HP", "categoria": "Electronica", 
     "precio": "15000.00", "stock": "5", "stock_minimo": "10"},
    {"sku": "SKU002", "nombre": "Mouse Logitech", "categoria": "Accesorios", 
     "precio": "350.00", "stock": "3", "stock_minimo": "15"},
    {"sku": "SKU003", "nombre": "Teclado Mecanico", "categoria": "Accesorios", 
     "precio": "800.00", "stock": "20", "stock_minimo": "10"},
    {"sku": "SKU004", "nombre": "Monitor LG", "categoria": "Electronica", 
     "precio": "6000.00", "stock": "8", "stock_minimo": "5"},
    {"sku": "SKU005", "nombre": "Audifonos Sony", "categoria": "Accesorios", 
     "precio": "1200.00", "stock": "2", "stock_minimo": "10"},
    {"sku": "SKU006", "nombre": "Webcam HD", "categoria": "Accesorios", 
     "precio": "450.00", "stock": "25", "stock_minimo": "20"},
    {"sku": "SKU007", "nombre": "SSD 1TB", "categoria": "Almacenamiento", 
     "precio": "1800.00", "stock": "0", "stock_minimo": "5"},
    # Registro invalido para probar validacion
    {"sku": "", "nombre": "Invalido", "categoria": "Test", 
     "precio": "abc", "stock": "-1", "stock_minimo": "5"},
]

print("=" * 60)
print("SISTEMA DE INVENTARIO - Reporte de Reorden")
print("=" * 60)

# Crear productos
productos = []
for datos in datos_inventario:
    es_valido, error = validar_producto(
        datos['sku'], datos['nombre'], datos['categoria'],
        datos['precio'], datos['stock'], datos['stock_minimo']
    )
    
    if not es_valido:
        print(f"[IGNORADO] {error}")
        continue
    
    p = Producto(
        datos['sku'], datos['nombre'], datos['categoria'],
        float(datos['precio']), int(datos['stock']), int(datos['stock_minimo'])
    )
    productos.append(p)

print(f"\nProductos validos: {len(productos)}")

# Filtrar y ordenar
necesitan_reorden = [p for p in productos if p.necesita_reorden()]
necesitan_reorden.sort(key=lambda p: p.unidades_faltantes(), reverse=True)

print(f"Necesitan reorden: {len(necesitan_reorden)}")

# Mostrar reporte
print("\n" + "-" * 60)
print("REPORTE DE PRODUCTOS QUE NECESITAN REORDEN")
print("-" * 60)
for p in necesitan_reorden:
    print(f"{p.sku}: {p.nombre}")
    print(f"   Stock: {p.stock}/{p.stock_minimo} | Faltan: {p.unidades_faltantes()} | Valor: ${p.valor_inventario():,.2f}")
    print()

# CSV de salida
print("-" * 60)
print("CSV DE SALIDA:")
print("-" * 60)
print("sku,nombre,categoria,stock_actual,stock_minimo,unidades_faltantes,valor_inventario")
for p in necesitan_reorden:
    print(f"{p.sku},{p.nombre},{p.categoria},{p.stock},{p.stock_minimo},{p.unidades_faltantes()},{p.valor_inventario():.2f}")

---

## Diagrama de Dependencias

```
╔════════════════════════════════════════════════════════════════════════════╗
║                     DIAGRAMA DE MODULOS                                    ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║                          ┌─────────────┐                                   ║
║                          │   main.py   │                                   ║
║                          └──────┬──────┘                                   ║
║                                 │                                          ║
║              ┌──────────────────┼──────────────────┐                       ║
║              │                  │                  │                       ║
║              ▼                  ▼                  ▼                       ║
║    ┌─────────────────┐  ┌─────────────┐  ┌─────────────────┐              ║
║    │ models/producto │  │  utils/io   │  │utils/validators │              ║
║    │                 │  │             │  │                 │              ║
║    │ class Producto  │  │ leer_csv()  │  │ validar_sku()   │              ║
║    │ - necesita_re.. │  │ escribir_.. │  │ validar_precio()│              ║
║    │ - valor_inven.. │  │             │  │ validar_stock() │              ║
║    └─────────────────┘  └─────────────┘  └─────────────────┘              ║
║                                                                            ║
║  FLUJO DE DATOS:                                                           ║
║  data/inventario.csv  ──>  main.py  ──>  outputs/reporte.csv               ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
```

---

## Entregables

### Estructura de Repositorio

```
reto-semana-04/
├── main.py
├── README.md
├── .gitignore
├── models/
│   ├── __init__.py
│   └── producto.py
├── utils/
│   ├── __init__.py
│   ├── io.py
│   └── validators.py
├── data/
│   └── inventario.csv
└── outputs/
    └── reporte_inventario.csv
```

### Criterios de Evaluacion

| Criterio | Puntos |
|----------|--------|
| Estructura de carpetas correcta | 15 |
| Clase Producto con todos los metodos | 25 |
| Validadores funcionando | 15 |
| Lectura/escritura de archivos | 15 |
| Reporte correcto (filtrado y ordenado) | 15 |
| Codigo modular y limpio | 10 |
| Git (commits, README) | 5 |
| **Total** | **100** |

### README.md Requerido

```markdown
# Sistema de Inventario Modular

## Descripcion
Sistema que genera reportes de productos que necesitan reorden.

## Estructura del Proyecto
[Describir carpetas y archivos]

## Como Ejecutar
```bash
python main.py
```

## Entrada
[Describir formato de data/inventario.csv]

## Salida
[Describir formato de outputs/reporte_inventario.csv]

## Autor
[Tu nombre]
```

---

## Fecha de Entrega

**Viernes de la Semana 4, 23:59 hrs**

---

*Reto Semana 4 - Programacion para Ciencia de Datos - IPN 2026*